# Exercício Prático Avaliado - Análise de Dados com PySpark
**Objetivo:** Utilizar Apache Spark (PySpark) para realizar a ingestão, tratamento e análise exploratória do dataset "USA.gov Data from Bitly".

## Célula 1: Instalação das dependências

Baixa o Java, o Findspark, Spark e o dataset de exemplo.

In [1]:
# Fazendo o download do Java e do Spark
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz
!tar xf spark-3.5.1-bin-hadoop3.tgz
!pip install -q findspark
!pip install -q pyspark

# Baixando o dataset "USA.gov Data from Bitly" (conhecido em datasets públicos)
!wget -q https://raw.githubusercontent.com/wesm/pydata-book/2nd-edition/datasets/bitly_usagov/example.txt -O bitly_data.json

## Célula 2: Configuração e Inicialização da SparkSession

In [2]:
from pyspark.sql import SparkSession

# Inicializando a SparkSession de forma direta
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("AnaliseBitly") \
    .getOrCreate()

print("SparkSession inicializada com sucesso!")

SparkSession inicializada com sucesso!


## Célula 3: Carregamento e Schema

In [ ]:
# A) Carrega o arquivo de dados em um DataFrame
df = spark.read.json("bitly_data.json")

# B) Exibe o schema do DataFrame
print("--- Schema do DataFrame ---")
df.printSchema()

# C) Mostrando as 10 primeiras linhas
print("--- 10 Primeiras Linhas ---")
df.show(10, truncate=False)

# Quantos registros existem no total?
total_registros = df.count()
print(f"Total de registros: {total_registros}")

--- Schema do DataFrame ---
root
 |-- _heartbeat_: long (nullable = true)
 |-- a: string (nullable = true)
 |-- al: string (nullable = true)
 |-- c: string (nullable = true)
 |-- cy: string (nullable = true)
 |-- g: string (nullable = true)
 |-- gr: string (nullable = true)
 |-- h: string (nullable = true)
 |-- hc: long (nullable = true)
 |-- hh: string (nullable = true)
 |-- kw: string (nullable = true)
 |-- l: string (nullable = true)
 |-- ll: array (nullable = true)
 |    |-- element: double (containsNull = true)
 |-- nk: long (nullable = true)
 |-- r: string (nullable = true)
 |-- t: long (nullable = true)
 |-- tz: string (nullable = true)
 |-- u: string (nullable = true)

--- 10 Primeiras Linhas ---
+-----------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------+----+----------+------+----+------+----------+

## Célula 4: Análise de Fuso Horário - Timezone

In [ ]:
from pyspark.sql.functions import col, desc

# Filtrando valores nulos ou vazios no fuso horário (tz)
df_tz_validos = df.filter(col("tz").isNotNull() & (col("tz") != ""))

# A) Quantos fusos horários distintos existem?
fusos_distintos = df_tz_validos.select("tz").distinct().count()
print(f"Fusos horários distintos: {fusos_distintos}")

# B) 10 fusos horários mais comuns
print("--- 10 Fusos Horários Mais Comuns ---")
df_tz_validos.groupBy("tz").count().orderBy(desc("count")).show(10, truncate=False)

Fusos horários distintos: 96
--- 10 Fusos Horários Mais Comuns ---
+-------------------+-----+
|tz                 |count|
+-------------------+-----+
|America/New_York   |1251 |
|America/Chicago    |400  |
|America/Los_Angeles|382  |
|America/Denver     |191  |
|Europe/London      |74   |
|Asia/Tokyo         |37   |
|Pacific/Honolulu   |36   |
|Europe/Madrid      |35   |
|America/Sao_Paulo  |33   |
|Europe/Berlin      |28   |
+-------------------+-----+
only showing top 10 rows


## Célula 5: Análise de Agentes de Usuário - User-Agent

In [ ]:
from pyspark.sql.functions import split

# A) Criando a nova coluna 'navegador' pegando a primeira parte da string antes da barra "/"
df = df.withColumn("navegador", split(col("a"), "/")[0])

# B) Contagem e os 5 mais frequentes
print("--- 5 Navegadores Mais Frequentes ---")
df.groupBy("navegador").count().orderBy(desc("count")).show(5, truncate=False)

--- 5 Navegadores Mais Frequentes ---
+-------------------+-----+
|navegador          |count|
+-------------------+-----+
|Mozilla            |3201 |
|GoogleMaps         |121  |
|NULL               |120  |
|Opera              |38   |
|TEST_INTERNET_AGENT|24   |
+-------------------+-----+
only showing top 5 rows


## Célula 6: Análise de Países

In [ ]:
# A) Qual a sigla do país (c) que mais gerou acessos?
print("--- País que mais gerou acessos ---")
df.filter(col("c").isNotNull() & (col("c") != "")).groupBy("c").count().orderBy(desc("count")).show(1)

# B) Primeiros 20 acessos que vieram do Brasil (BR)
print("--- Primeiros 20 acessos do Brasil (BR) ---")
df.filter(col("c") == "BR").select("tz", "u").show(20, truncate=False)

--- País que mais gerou acessos ---
+---+-----+
|  c|count|
+---+-----+
| US| 2305|
+---+-----+
only showing top 1 row
--- Primeiros 20 acessos do Brasil (BR) ---
+-----------------+-------------------------------------------------------------------+
|tz               |u                                                                  |
+-----------------+-------------------------------------------------------------------+
|America/Sao_Paulo|http://apod.nasa.gov/apod/ap120312.html                            |
|America/Sao_Paulo|http://apod.nasa.gov/apod/ap120312.html                            |
|America/Sao_Paulo|http://apod.nasa.gov/apod/ap120312.html                            |
|America/Sao_Paulo|http://apod.nasa.gov/apod/ap120312.html                            |
|America/Sao_Paulo|http://www.nasa.gov/mission_pages/nustar/main/index.html           |
|America/Sao_Paulo|http://www.nasa.gov/mission_pages/WISE/multimedia/pia15481.html    |
|America/Recife   |http://apod.nasa.gov/apod/

## Célula 7: Análise de Sistemas Operacionais

In [ ]:
from pyspark.sql.functions import when

# A) Criando a coluna sistema_operacional usando Regex com 'rlike' (case-insensitive "(?i)")
df = df.withColumn("sistema_operacional",
    when(col("a").rlike("(?i)windows"), "Windows")
    .when(col("a").rlike("(?i)mac"), "Mac")
    .when(col("a").rlike("(?i)linux"), "Linux")
    .otherwise("Outro")
)

# B) Contagem de acessos por sistema_operacional
print("--- Contagem por Sistema Operacional ---")
df.groupBy("sistema_operacional").count().orderBy(desc("count")).show()

--- Contagem por Sistema Operacional ---
+-------------------+-----+
|sistema_operacional|count|
+-------------------+-----+
|            Windows| 2246|
|                Mac|  773|
|              Outro|  375|
|              Linux|  166|
+-------------------+-----+



## Parte 3 - Dificuldades Encontradas

Neste exercício prático, o principal desafio foi **[compreender como manipular strings dentro da sintaxe do PySpark (ainda mais na célula de navegadores e sistemas operacionais), além do processo de limpar adequadamente valores nulos antes de agregar as contagens.]**

A configuração inicial do ambiente no Colab, garante as versões corretas do Java e do Spark para que a `SparkSession` funcione corretamente.

---
**Nota sobre o uso de Inteligência Artificial:**
Utilizei a inteligência artificial (ChatGPT/Gemini) como um assistente de aprendizado durante este projeto. O uso limitou-se a [Tirei dúvidas também sobre a aplicação da função regex (`rlike`) na criação da coluna de Sistemas Operacionais e para estruturar de maneira padrão para as funções de agrupamento e agregação (`groupBy` e `count`) no PySpark]. Todo o código gerado foi revisado e testado no meu ambiente para garantir que eu compreendi a lógica por trás de cada linha utilizada.